# Step 1 Inventory: RBH
Profiles every file at the site (columns, types, rows, ID and date formats) and records what later steps must be told: candidate keys, files that look like copies of one another, free-text columns, and years where a file's rows collapse. Nothing is changed. Outputs go to `derived/` and are counts, shapes and column names only.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.home() / "Documents/DS217/users/jormond/open_picu/code"))
import inventory as I

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 200)

In [ ]:
site = "RBH"                      # "SMH" | "GOSH"
demo = "ai_picu_demographics.csv" # GOSH: "PICANet_ALL_patients_SR_011024_projectID_eprweights.csv"
exclude = None                    # GOSH: r"lab_components"

base = Path.home() / "Documents/DS217/users/jormond"
open_picu_dir = base / "open_picu"
derived_dir = open_picu_dir / "derived"
reports_dir = derived_dir / "reports"
site_dir = base / site

# GOSH only: I.list_files(site_dir) first, then add recursive=True, n_jobs=3
inv = I.inventory_site(site_dir, exclude=exclude)
I.save_inventory(inv, derived_dir / "inventory" / f"{site.lower()}_inventory.pkl")
inv["summary"]

## 1. Structure
Files with identical columns are grouped (one file per variable is common at GOSH). Candidate keys are every ID-like column; "unique per row" tells you which could identify a row. ID overlap is the share of each file's IDs that appear in the base file.

In [ ]:
I.schema_groups(inv)

In [ ]:
I.id_overlap(inv, base=demo)

## 2. Dates and eras
Format, range and resolution of every date column, then rows per calendar year by each file's main time column. `find_gaps` lists years below half of the file's usual level. These are leads to explain (system changeover, outage, pandemic), not errors. `rows_by_year` rereads one column per file, so allow a few minutes at GOSH.

In [ ]:
inv["dates"]

In [ ]:
by_year = I.rows_by_year(inv)
by_year

In [ ]:
I.find_gaps(by_year)

## 3. Privacy flags and look-alike files
Columns flagged as free text or direct identifiers are carried forward: no later step may release a column with one of these names without a decision. Look-alike files are candidates to settle at step 2.

In [ ]:
I.identifier_audit(inv)

In [ ]:
I.lookalike_files(inv)

## 4. Decisions and notes
The only place judgements are typed. `rejected` lists files that must not be used and why. `notes` are short statements, with no values or IDs, that later steps should know.

In [ ]:
# RBH
rejected = {"ai_picu_piccanet_old.csv": "old PICANet file; not used (manifest rule 2)"}
notes = ["studyID = prefix (NIC/NND) : PATIENT_RK : SAS-second admission time; 5 stays appear under both prefixes",
         "NND rows admitted before 2010 mostly have no vitals (7,917 empty studyIDs)",
         "demographics holds 14,797 studyIDs, 1,139 admissions have more than one studyID"]

# SMH
# rejected = {}
# notes = ["encounterId is the stay key; UniquePtID is not equal between demographics and piccanet, so do not link on it",
#          "large exact-duplicate row counts in vitals, vent, airway, io (see summary)"]

# GOSH
# rejected = {"Archived/picanet_file_with_mapping.csv": "unreliable discharge status; not used",
#             "PICANet_All_patients_SR_011024_projectID.csv": "identical to the eprweights file; eprweights used"}
# notes = ["links by project_id; admission_identifier on about 56% of HR rows only",
#          "Epic era from 2019-04-20: see GOSH_EPIC_ERA.md"]

In [ ]:
paths = I.finalise(inv, site, reports_dir, base=demo, by_year=by_year,
                   rejected=rejected, excluded=exclude, notes=notes)
paths

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
sys.path.append(str(Path.home() / "Documents/DS217/users/jormond/open_picu"))
import pandas as pd
import code.inventory as I
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 200)

In [ ]:
base = Path.home() / "Documents/DS217/users/jormond"
open_picu_dir = base / "open_picu"
rbh_dir = base / "RBH"

rbh = I.inventory_site(rbh_dir)   # add max_rows=2_000_000 for a first pass
rbh["summary"]

In [ ]:
rbh["dates"]  

In [ ]:
I.identifier_audit(rbh)

In [ ]:
I.write_report(rbh, "RBH", open_picu_dir/"RBH_inventory.md")

In [ ]:
v = pd.read_csv(rbh_dir/"ai_picu_vitals.csv",
                usecols=["studyID","PAS_AD_RK","chartTime","WARD_IN_dttm","WARD_OUT_dttm",
                         "intv_hr","Day","LOS","clinicalUnit"])
for c in ["chartTime","WARD_IN_dttm","WARD_OUT_dttm"]:
    v[c] = I.parse_dates(v[c])[0]

hrs = (v["chartTime"] - v["WARD_IN_dttm"]).dt.total_seconds() / 3600
print("intv_hr minus hours since WARD_IN:\n", (v["intv_hr"] - hrs).describe())
stay = v.drop_duplicates("studyID")
los_h = (stay["WARD_OUT_dttm"] - stay["WARD_IN_dttm"]).dt.total_seconds() / 3600
print("LOS / (WARD_OUT - WARD_IN in hours):\n", (stay["LOS"] / los_h).describe())
print("WARD_OUT before WARD_IN:", int((los_h < 0).sum()))
print(v["chartTime"].dt.year.value_counts().sort_index())

In [ ]:
g = v.groupby("PAS_AD_RK").agg(n_study=("studyID","nunique"),
                               n_in=("WARD_IN_dttm","nunique"),
                               n_out=("WARD_OUT_dttm","nunique"))
multi = g[g.n_study > 1]
print(len(multi), "admissions with more than one studyID")
print("distinct WARD_IN per such admission:\n", multi["n_in"].value_counts().head())
print("distinct WARD_OUT per such admission:\n", multi["n_out"].value_counts().head())

s = (v.groupby(["PAS_AD_RK","studyID"])["chartTime"].agg(["min","max"])
       .reset_index().sort_values(["PAS_AD_RK","min"]))
s["prev_max"] = s.groupby("PAS_AD_RK")["max"].shift()
print("studyIDs starting before the previous one ended:", int((s["min"] < s["prev_max"]).sum()))
print("gap to previous studyID (hours):\n",
      ((s["min"] - s["prev_max"]).dt.total_seconds() / 3600).describe())

In [ ]:
has_picu = v["clinicalUnit"].eq("RBH-PICU").groupby(v["studyID"]).any()
pn_new = set(pd.read_csv(rbh_dir/"ai_picu_piccanet.csv", usecols=["studyID"])["studyID"])
pn_old = set(pd.read_csv(rbh_dir/"ai_picu_piccanet_old.csv", usecols=["studyID"])["studyID"])
print(pd.crosstab(has_picu.rename("any PICU row"),
                  has_picu.index.isin(pn_new), rownames=["any PICU row"], colnames=["in PICANet new"]))
print("in PICANet old:", has_picu.index.isin(pn_old).sum())
print("PICANet new IDs with no PICU row:", len(pn_new - set(has_picu[has_picu].index)))

In [ ]:
d = pd.read_csv(rbh_dir/"ai_picu_demographics.csv")
dup = d[d["studyID"].duplicated(keep=False)]
print(len(dup), "rows;", dup["studyID"].nunique(), "studyIDs")
print("columns that differ within a studyID:\n", dup.groupby("studyID").nunique().max())

In [ ]:
d = pd.read_csv(rbh_dir/"ai_picu_demographics.csv")
parts = d["studyID"].str.split(":", expand=True)
print(parts[0].value_counts())
mid, last = parts[1].astype("int64"), parts[2].astype("int64")
print("middle == PATIENT_RK:", (mid == d["PATIENT_RK"]).mean())
print("middle == PAS_AD_RK:", (mid == d["PAS_AD_RK"]).mean())
t = I.parse_dates(d["PICUadm_dttm"])[0]
for name, epoch in [("SAS", "1960-01-01"), ("unix", "1970-01-01")]:
    ref = (t - pd.Timestamp(epoch)).dt.total_seconds()
    print(name, "last part minus PICUadm_dttm (s):\n", (last - ref).abs().describe())

In [ ]:
n_rows = v.groupby("studyID")["chartTime"].count()
print("studyIDs with no timestamped vitals rows:", int((n_rows == 0).sum()))

ad = v.drop_duplicates("studyID").set_index("studyID")["PAS_AD_RK"]
picu_ids = has_picu[has_picu].index
adm = pd.DataFrame({"PAS_AD_RK": ad,
                    "picu": ad.index.isin(picu_ids),
                    "picanet": ad.index.isin(pn_new)})
a = adm.groupby("PAS_AD_RK")[["picu", "picanet"]].any()
print(pd.crosstab(a["picu"], a["picanet"], rownames=["any PICU row"], colnames=["in PICANet new"]))

In [ ]:
d["prefix"]  = d["studyID"].str.split(":").str[0]
d["empty"]   = d["studyID"].map(n_rows).eq(0)
d["has_picu"] = d["studyID"].map(has_picu)
d["picanet"] = d["studyID"].isin(pn_new)
d["year"]    = I.parse_dates(d["PICUadm_dttm"])[0].dt.year

print(d.groupby("prefix")[["empty", "has_picu", "picanet"]].agg(["sum", "mean"]).round(2))
print(pd.crosstab(d["year"], d["empty"]))
print(d.groupby("empty")[["Age_mth", "Weight_kg"]].describe().T)

In [ ]:
d[["studyID", "prefix", "empty", "has_picu", "picanet", "year"]].to_csv(
    open_picu_dir/"RBH_cohort_flags.csv", index=False)

In [ ]:
sed = pd.read_csv(rbh_dir/"ai_picu_sedation.csv",
                  usecols=lambda c: c.lower() in ("studyid", "patient_rk", "pas_ad_rk"))
sed.columns = sed.columns.str.lower()
ids = sed.drop_duplicates()
out = ids[~ids["studyid"].isin(d["studyID"])]

print("unique studyIDs:", ids["studyid"].nunique(),
      "| not in cohort:", out["studyid"].nunique())
print("  with missing patient_rk:", out.loc[out["patient_rk"].isna(), "studyid"].nunique())
print("  with patient_rk in cohort:", out.loc[out["patient_rk"].isin(d["PATIENT_RK"]), "studyid"].nunique())
print("  with pas_ad_rk in cohort:", out.loc[out["pas_ad_rk"].isin(d["PAS_AD_RK"]), "studyid"].nunique())
print("  prefixes:\n", out["studyid"].str.split(":").str[0].value_counts())
print("rows with missing patient_rk overall:", round(sed["patient_rk"].isna().mean(), 3))

In [ ]:
out = out.copy()
out["t"] = pd.Timestamp("1960-01-01") + pd.to_timedelta(
    out["studyid"].str.split(":").str[2].astype("int64"), unit="s")
print(out["t"].dt.year.value_counts().sort_index())

d["t"] = I.parse_dates(d["PICUadm_dttm"])[0]
m = out.merge(d[["PAS_AD_RK", "t"]], left_on="pas_ad_rk", right_on="PAS_AD_RK",
              suffixes=("", "_coh"))
m["dh"] = (m["t"] - m["t_coh"]).dt.total_seconds() / 3600
print("hours to nearest cohort studyID in the same admission:\n",
      m.groupby("studyid")["dh"].apply(lambda x: x.abs().min()).describe())

In [ ]:
mm = out.merge(d[["studyID", "PAS_AD_RK", "t"]], left_on="pas_ad_rk", right_on="PAS_AD_RK",
               suffixes=("", "_coh"))
same = mm[(mm["t"] - mm["t_coh"]).abs() < pd.Timedelta(seconds=1)]
print(same["studyid"].nunique(), "out-of-cohort studyIDs with a cohort studyID at the identical time")
print(same["studyID"].str.split(":").str[0].value_counts())   # prefix of the cohort twin
print("same patient part:",
      (same["studyid"].str.split(":").str[1] == same["studyID"].str.split(":").str[1]).mean())

In [ ]:
nz = d[~d["empty"]]
key = nz["PATIENT_RK"].astype(str) + "|" + nz["PICUadm_dttm"].astype(str)
print("non-empty studyIDs:", nz["studyID"].nunique(), "| distinct stays (patient + time):", key.nunique())
print("prefixes per stay:\n", nz.groupby(key)["prefix"].nunique().value_counts())

sed_ids = ids["studyid"].drop_duplicates()
pref = sed_ids.str.split(":").str[0]
print(pd.crosstab(pref, sed_ids.isin(d["studyID"]), colnames=["in demographics"]))

In [ ]:
sed = pd.read_csv(rbh_dir/"ai_picu_sedation.csv",
                  usecols=lambda c: c.lower() in ("studyid", "charttime", "drug", "dose"))
sed.columns = sed.columns.str.lower()
p = sed["studyid"].str.split(":", expand=True)
sed["prefix"], sed["stay"] = p[0], p[1] + ":" + p[2]

both = sed.groupby("stay")["prefix"].nunique()
both = both[both == 2].index
print("stays present under both prefixes in sedation:", len(both))

sb = sed[sed["stay"].isin(both)]
n = sb.groupby(["stay", "prefix"]).size().unstack()
print("rows per copy:\n", n.describe())
print("stays with equal row counts in both copies:", round((n["NIC"] == n["NND"]).mean(), 3))

cols = ["stay", "charttime", "drug", "dose"]
a = sb.loc[sb.prefix == "NIC", cols].drop_duplicates()
b = sb.loc[sb.prefix == "NND", cols].drop_duplicates()
print("identical records in both copies:", len(a.merge(b, on=cols)), "of", len(a), "(NIC) and", len(b), "(NND)")

In [ ]:
for f in sorted(rbh_dir.glob("*.csv")):
    ids = pd.read_csv(f, usecols=lambda c: c.lower() == "studyid")
    if ids.empty:
        continue
    u = ids.iloc[:, 0].dropna().drop_duplicates().str.split(":", expand=True)
    stay = u[1] + ":" + u[2]
    n_both = int((u.groupby(stay)[0].nunique() == 2).sum())
    print(f"{f.name}: {len(u)} studyIDs | {stay.nunique()} stays | {n_both} under both prefixes")

In [ ]:
def stays(f):
    ids = pd.read_csv(f, usecols=lambda c: c.lower() == "studyid").iloc[:, 0].dropna().drop_duplicates()
    p = ids.str.split(":", expand=True)
    return set(p[1] + ":" + p[2])

new = stays(rbh_dir/"ai_picu_piccanet.csv")
old = stays(rbh_dir/"ai_picu_piccanet_old.csv")
coh = stays(rbh_dir/"ai_picu_demographics.csv")
print("new:", len(new), "old:", len(old), "| both:", len(new & old),
      "| new only:", len(new - old), "| old only:", len(old - new))
print("old only, in cohort:", len((old - new) & coh), "| not in cohort:", len((old - new) - coh))
print("share of new in cohort:", round(len(new & coh) / len(new), 3))

In [ ]:
def years(keys):
    secs = pd.Series([int(k.split(":")[1]) for k in keys])
    return (pd.Timestamp("1960-01-01") + pd.to_timedelta(secs, unit="s")).dt.year

for name, keys in [("old only, in cohort", (old - new) & coh),
                   ("old only, not in cohort", (old - new) - coh),
                   ("new only", new - old)]:
    print(name, len(keys)); print(years(keys).value_counts().sort_index().to_string())

In [ ]:
pn = pd.read_csv(rbh_dir/"ai_picu_piccanet.csv", usecols=["studyID"])["studyID"].drop_duplicates()
has_data = n_rows.reindex(pn).fillna(0) > 0
print("PICANet-new studyIDs:", len(pn),
      "| with vitals rows:", int(has_data.sum()),
      "| with a PICU row:", int(has_picu.reindex(pn).fillna(False).sum()),
      "| no vitals at all:", int((~has_data).sum()))

secs = pn[~has_data.values].str.split(":").str[2].astype("int64")
print("no-vitals PICANet studyIDs by admission year:")
print((pd.Timestamp("1960-01-01") + pd.to_timedelta(secs, unit="s")).dt.year.value_counts().sort_index().to_string())

In [ ]:
vit = n_rows[n_rows > 0].index
rest = pd.Index(vit).difference(pn)
print(len(rest), "studyIDs with vitals not in PICANet-new | with a PICU row:",
      int(has_picu.reindex(rest).sum()))
print(pd.Series(rest.str.split(":").str[0]).value_counts())
secs = pd.Series(rest.str.split(":").str[2].astype("int64"))
print((pd.Timestamp("1960-01-01") + pd.to_timedelta(secs, unit="s")).dt.year.value_counts().sort_index().to_string())

In [ ]:
tw = same["studyid"].unique()
rest = m[~m["studyid"].isin(tw)]
g = rest.groupby("studyid")["dh"].apply(lambda x: x.abs().min())
print(len(g), "studyIDs; hours to nearest cohort studyID in the same admission:")
print(g.describe())
print((g < 24).sum(), "are within 24 h")

In [ ]:
# 1. ID hierarchy
demo = pd.read_csv(rbh_dir / "ai_picu_demographics.csv")
print(demo.groupby("PAS_AD_RK")["studyID"].nunique().value_counts().sort_index())
print(demo.groupby("PATIENT_RK")["studyID"].nunique().describe())
print(demo[demo.duplicated("studyID", keep=False)].shape)

# 2. PICANet old vs new
pn, pno = (pd.read_csv(rbh_dir / f) for f in ["ai_picu_piccanet.csv", "ai_picu_piccanet_old.csv"])
s_new, s_old, s_all = set(pn.studyID), set(pno.studyID), set(demo.studyID)
print(len(s_new & s_old), len(s_new - s_old), len(s_old - s_new), len(s_new - s_all), len(s_old - s_all))

# 3. Timestamp formats (first rows)
pd.read_csv(rbh_dir / "ai_picu_vitals.csv", nrows=5)[["chartTime", "WARD_IN_dttm", "WARD_OUT_dttm", "Day", "intv_hr"]]

# 4. Ventilation modes and charting interval
vent = pd.read_csv(rbh_dir / "ai_picu_vent.csv", usecols=["studyID", "intv_hr", "Ventilation_Mode"])
print(vent["Ventilation_Mode"].value_counts(dropna=False))
gap = vent.sort_values(["studyID", "intv_hr"]).groupby("studyID")["intv_hr"].diff()
print(gap.describe(percentiles=[.1, .5, .9, .99]))

In [ ]:
rbh["profiles"]["ai_picu_vitals.csv"]

In [ ]:
I.shared_columns(rbh)

In [ ]:
I.id_overlap(rbh)

In [ ]:
I.write_report(rbh, "RBH", open_picu_dir / "RBH_inventory.md")

In [ ]:
b = pd.read_csv(rbh_dir/"ai_picu_biochemistry.csv", usecols=["studyID", "SampleCollectDateTime"])
lab_ids = set(b.loc[b["SampleCollectDateTime"].notna(), "studyID"])
vit_ids = set(n_rows[n_rows > 0].index)
only = lab_ids - vit_ids
print(len(lab_ids), "studyIDs with biochemistry |", len(only), "of them have no vitals")
secs = pd.Series([int(k.split(":")[2]) for k in only])
print((pd.Timestamp("1960-01-01") + pd.to_timedelta(secs, unit="s")).dt.year.value_counts().sort_index().to_string())

In [ ]:
d = pd.read_csv(rbh_dir/"ai_picu_demographics.csv", usecols=["studyID", "PICUadm_dttm"])
d["adm"] = pd.to_datetime(d["PICUadm_dttm"], format="%d%b%Y:%H:%M:%S.%f")
d["has_vitals"] = d["studyID"].isin(vit_ids)
m = d[(d.adm >= "2008-10-01") & (d.adm < "2011-01-01")]
print(m.groupby([m.adm.dt.to_period("M"), "has_vitals"]).size().unstack(fill_value=0).to_string())

In [ ]:
for f in ["infusion", "meds", "sedation"]:
    x = pd.read_csv(rbh_dir/f"ai_picu_{f}.csv", usecols=["studyID", "PAS_AD_RK", "encounterId"]).drop_duplicates()
    print(f, "| encounterIds per PAS_AD_RK (max):", x.groupby("PAS_AD_RK")["encounterId"].nunique().max(),
          "| PAS_AD_RK per encounterId (max):", x.groupby("encounterId")["PAS_AD_RK"].nunique().max(),
          "| studyIDs per encounterId (max):", x.groupby("encounterId")["studyID"].nunique().max(),
          "| range:", x["encounterId"].min(), "to", x["encounterId"].max())